# BERT Encoder 预训练从零实现：客服工单 MLM

## 面试问题

面试时我会说明 BERT Encoder 使用双向自注意力，每个位置都能读取左右上下文；MLM 把部分输入替换为 `[MASK]`，只在被选位置计算原 token 的预测损失。与 GPT 不同，它不使用因果 mask。位置 embedding 与 token embedding 相加后进入注意力、残差、LayerNorm 和前馈层。训练数据制作必须防止把原 token 留在输入中造成标签泄漏，评估也要按被 mask 位置计算准确率。原始 BERT 还使用 NSP，但许多后续方案省略或替换它；本例只实现 MLM，绝不把缺失的 NSP 冒充已实现。下面手写 Encoder 与 MLM 头，展示双向注意力、候选 logits 和泄漏修复。

## 真实案例

数据是八条脱敏客服工单，每条七个 token，覆盖登录、锁定、发货、物流延迟、发票、库存、扣款和优惠。每条都遮住索引 3 的故障动作词，目标是依据左右上下文恢复它。数据为封闭小语料并在全样本上拟合，只验证 MLM 数据流与 Encoder 机制。

本实验是为了看清机制而构造的离线小样本，不代表线上收益，也不能外推到开放分布。

In [1]:
import math  # 导入平方根用于缩放点积注意力。
from collections import Counter  # 导入计数器实现最频繁 token 基线。
import torch  # 导入 PyTorch 以实现双向 Encoder 和 MLM。
from torch import nn  # 导入神经网络基础层。
import torch.nn.functional as F  # 导入 GELU、softmax 和交叉熵。
torch.manual_seed(32)  # 固定随机种子以复现实验输出。
torch.set_num_threads(1)  # 限制 CPU 线程以稳定小实验运行。
ticket_sequences = [["工单", "用户", "无法", "登录", "请", "重置", "密码"], ["工单", "账号", "突然", "锁定", "请", "核验", "身份"], ["工单", "订单", "一直", "待发", "请", "查询", "物流"], ["工单", "包裹", "到货", "延迟", "请", "查询", "物流"], ["工单", "电子", "发票", "错误", "请", "修改", "抬头"], ["工单", "商品", "显示", "缺货", "请", "检查", "库存"], ["工单", "支付", "重复", "扣款", "请", "发起", "退款"], ["工单", "优惠", "无法", "使用", "请", "核对", "活动"]]  # 定义八条可读客服工单序列。
vocabulary = ["[PAD]", "[MASK]"] + sorted({token for sequence in ticket_sequences for token in sequence})  # 构造含特殊 token 的封闭词表。
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 建立 token 到编号的映射。
id_to_token = {index: token for token, index in token_to_id.items()}  # 建立编号到 token 的反向映射。
original_ids = torch.tensor([[token_to_id[token] for token in sequence] for sequence in ticket_sequences], dtype=torch.long)  # 编码八条未遮盖工单。
mask_position = 3  # 固定遮盖每条工单的故障动作位置。
mlm_targets = original_ids[:, mask_position].clone()  # 保存被遮盖位置的原 token 作为 MLM 标签。
masked_ids = original_ids.clone()  # 复制原始 token 张量以制作模型输入。
masked_ids[:, mask_position] = token_to_id["[MASK]"]  # 用统一的 MASK 编号替换目标位置。
print("序号  原工单                              MLM 输入")  # 打印真实工单与遮盖输入表头。
for index, sequence in enumerate(ticket_sequences):  # 逐条展示人类可读的 MLM 数据制作结果。
    masked_tokens = sequence.copy()  # 复制当前工单 token 以便可读展示。
    masked_tokens[mask_position] = "[MASK]"  # 在展示序列中替换目标词。
    print(f"{index + 1:02d}    {'/'.join(sequence):<32}  {'/'.join(masked_tokens)}")  # 输出原序列与模型实际输入。
print(f"输入形状={tuple(masked_ids.shape)}，词表大小={len(vocabulary)}，MLM 样本={len(ticket_sequences)}")  # 汇总数据规模和张量形状。

序号  原工单                              MLM 输入
01    工单/用户/无法/登录/请/重置/密码               工单/用户/无法/[MASK]/请/重置/密码
02    工单/账号/突然/锁定/请/核验/身份               工单/账号/突然/[MASK]/请/核验/身份
03    工单/订单/一直/待发/请/查询/物流               工单/订单/一直/[MASK]/请/查询/物流
04    工单/包裹/到货/延迟/请/查询/物流               工单/包裹/到货/[MASK]/请/查询/物流
05    工单/电子/发票/错误/请/修改/抬头               工单/电子/发票/[MASK]/请/修改/抬头
06    工单/商品/显示/缺货/请/检查/库存               工单/商品/显示/[MASK]/请/检查/库存
07    工单/支付/重复/扣款/请/发起/退款               工单/支付/重复/[MASK]/请/发起/退款
08    工单/优惠/无法/使用/请/核对/活动               工单/优惠/无法/[MASK]/请/核对/活动
输入形状=(8, 7)，词表大小=41，MLM 样本=8


## 基线：所有 MASK 都猜训练目标中的最高频 token

八个目标词在本数据中各出现一次，稳定 tie-break 后基线只能碰巧命中一条。它与 Encoder 在完全相同的八个 MASK 位置上计算准确率。

In [2]:
target_counts = Counter(mlm_targets.tolist())  # 统计被遮盖目标 token 的训练频次。
most_frequent_target = min(target_counts, key=lambda token_id: (-target_counts[token_id], token_id))  # 按频次和稳定编号选择基线 token。
baseline_predictions = torch.full_like(mlm_targets, most_frequent_target)  # 对全部 MASK 位置预测同一个高频 token。
baseline_accuracy = (baseline_predictions == mlm_targets).float().mean().item()  # 计算八条工单上的基线准确率。
print(f"最高频基线统一预测={id_to_token[most_frequent_target]}")  # 展示基线实际猜测的 token。
print("工单目标=" + "、".join(id_to_token[index.item()] for index in mlm_targets))  # 展示八个真实 MLM 标签。
print(f"最高频 token 基线准确率={baseline_accuracy:.1%}")  # 汇总同数据上的基线指标。

最高频基线统一预测=使用
工单目标=登录、锁定、待发、延迟、错误、缺货、扣款、使用
最高频 token 基线准确率=12.5%


## 手写核心：无因果 mask 的双向自注意力与 MLM 头

下面不导入 `TransformerEncoder` 或现成 BERT。被 MASK 的位置可以对左侧“故障描述”和右侧“处理动作”同时分配注意力，这正是 Encoder 与自回归 Decoder 的关键差异。

In [3]:
class BidirectionalSelfAttention(nn.Module):  # 定义不使用因果 mask 的手写多头自注意力。
    def __init__(self, model_dim, head_count):  # 根据隐藏维度和头数创建投影层。
        super().__init__()  # 初始化父类以注册全部参数。
        self.head_count = head_count  # 保存注意力头数供张量重排。
        self.head_dim = model_dim // head_count  # 计算每个注意力头的维度。
        self.query_projection = nn.Linear(model_dim, model_dim, bias=False)  # 创建 Query 投影矩阵。
        self.key_projection = nn.Linear(model_dim, model_dim, bias=False)  # 创建 Key 投影矩阵。
        self.value_projection = nn.Linear(model_dim, model_dim, bias=False)  # 创建 Value 投影矩阵。
        self.output_projection = nn.Linear(model_dim, model_dim, bias=False)  # 创建多头输出投影矩阵。
    def forward(self, hidden):  # 对整条工单执行双向自注意力。
        batch_size, sequence_length, model_dim = hidden.shape  # 读取输入批次、序列长度和隐藏维度。
        query = self.query_projection(hidden).view(batch_size, sequence_length, self.head_count, self.head_dim).transpose(1, 2)  # 投影并重排 Query。
        key = self.key_projection(hidden).view(batch_size, sequence_length, self.head_count, self.head_dim).transpose(1, 2)  # 投影并重排 Key。
        value = self.value_projection(hidden).view(batch_size, sequence_length, self.head_count, self.head_dim).transpose(1, 2)  # 投影并重排 Value。
        scores = query @ key.transpose(-2, -1) / math.sqrt(self.head_dim)  # 计算所有位置两两之间的缩放点积分数。
        attention = torch.softmax(scores, dim=-1)  # 不施加因果 mask 并在全部位置上归一化。
        context = attention @ value  # 用双向注意力权重聚合上下文。
        merged = context.transpose(1, 2).contiguous().view(batch_size, sequence_length, model_dim)  # 把多个注意力头重新拼接。
        output = self.output_projection(merged)  # 混合各注意力头的上下文表示。
        return output, attention  # 返回上下文和可解释的双向注意力权重。
class BertEncoderBlock(nn.Module):  # 定义一个后归一化教学版 BERT Encoder Block。
    def __init__(self, model_dim, head_count):  # 创建注意力、归一化和前馈层。
        super().__init__()  # 初始化父类以注册子模块。
        self.attention = BidirectionalSelfAttention(model_dim, head_count)  # 创建手写双向自注意力。
        self.attention_norm = nn.LayerNorm(model_dim)  # 创建注意力残差后的 LayerNorm。
        self.feedforward_up = nn.Linear(model_dim, model_dim * 2)  # 把隐藏维度扩展两倍。
        self.feedforward_down = nn.Linear(model_dim * 2, model_dim)  # 把前馈激活映射回模型维度。
        self.feedforward_norm = nn.LayerNorm(model_dim)  # 创建前馈残差后的 LayerNorm。
    def forward(self, hidden):  # 完成一次双向注意力和前馈计算。
        attention_output, attention = self.attention(hidden)  # 计算全序列双向上下文。
        hidden = self.attention_norm(hidden + attention_output)  # 应用注意力残差和归一化。
        feedforward = self.feedforward_down(F.gelu(self.feedforward_up(hidden)))  # 计算两层 GELU 前馈网络。
        hidden = self.feedforward_norm(hidden + feedforward)  # 应用前馈残差和归一化。
        return hidden, attention  # 返回编码结果和注意力权重。
class TinyBertForMLM(nn.Module):  # 定义单层教学版 BERT MLM 模型。
    def __init__(self, vocabulary_size, model_dim=32, head_count=4, max_length=16):  # 创建 embedding、Encoder 和 MLM 头。
        super().__init__()  # 初始化父类以注册全部参数。
        self.token_embedding = nn.Embedding(vocabulary_size, model_dim)  # 创建 token embedding 表。
        self.position_embedding = nn.Embedding(max_length, model_dim)  # 创建绝对位置 embedding 表。
        self.segment_embedding = nn.Embedding(2, model_dim)  # 创建句段 embedding 以对应 BERT 输入结构。
        self.encoder = BertEncoderBlock(model_dim, head_count)  # 创建手写双向 Encoder Block。
        self.mlm_transform = nn.Linear(model_dim, model_dim)  # 创建 MLM 输出前的隐藏变换。
        self.mlm_norm = nn.LayerNorm(model_dim)  # 创建 MLM 输出前的归一化。
        self.mlm_head = nn.Linear(model_dim, vocabulary_size)  # 把 MASK 隐藏状态映射到词表 logits。
    def forward(self, input_ids):  # 对一批遮盖工单执行 MLM 前向传播。
        positions = torch.arange(input_ids.shape[1], device=input_ids.device)  # 创建每条工单共享的位置编号。
        segment_ids = torch.zeros_like(input_ids)  # 本例只有单句段所以句段编号全为零。
        hidden = self.token_embedding(input_ids) + self.position_embedding(positions).unsqueeze(0) + self.segment_embedding(segment_ids)  # 相加三种 BERT 输入 embedding。
        encoded, attention = self.encoder(hidden)  # 通过手写双向 Encoder 获取上下文表示。
        transformed = self.mlm_norm(F.gelu(self.mlm_transform(encoded)))  # 对编码结果执行 MLM 专用变换与归一化。
        logits = self.mlm_head(transformed)  # 输出每个位置的完整词表 logits。
        return logits, attention, encoded  # 返回词表预测、注意力和编码中间量。
bert = TinyBertForMLM(len(vocabulary))  # 实例化封闭词表上的教学版 BERT。
print(bert)  # 展示手写 Encoder 与 MLM 头的实际结构。
print(f"可训练参数量={sum(parameter.numel() for parameter in bert.parameters())}")  # 输出教学模型参数规模。

TinyBertForMLM(
  (token_embedding): Embedding(41, 32)
  (position_embedding): Embedding(16, 32)
  (segment_embedding): Embedding(2, 32)
  (encoder): BertEncoderBlock(
    (attention): BidirectionalSelfAttention(
      (query_projection): Linear(in_features=32, out_features=32, bias=False)
      (key_projection): Linear(in_features=32, out_features=32, bias=False)
      (value_projection): Linear(in_features=32, out_features=32, bias=False)
      (output_projection): Linear(in_features=32, out_features=32, bias=False)
    )
    (attention_norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (feedforward_up): Linear(in_features=32, out_features=64, bias=True)
    (feedforward_down): Linear(in_features=64, out_features=32, bias=True)
    (feedforward_norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  )
  (mlm_transform): Linear(in_features=32, out_features=32, bias=True)
  (mlm_norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (mlm_head): Linear(in_featu

In [4]:
optimizer = torch.optim.Adam(bert.parameters(), lr=0.03)  # 创建优化器学习八条工单的 MLM 映射。
loss_trace = []  # 保存 MLM 训练损失轨迹。
first_gradient_norm = 0.0  # 预留首轮 token embedding 梯度范数。
for epoch in range(401):  # 在受控小语料上执行四百零一次更新。
    optimizer.zero_grad()  # 清空上一轮累计梯度。
    all_logits, attention, encoded = bert(masked_ids)  # 对真正遮盖后的输入运行双向 Encoder。
    masked_logits = all_logits[:, mask_position, :]  # 只提取被 MASK 位置的词表 logits。
    loss = F.cross_entropy(masked_logits, mlm_targets)  # 只在选定 MLM 位置计算交叉熵。
    loss.backward()  # 反向传播到 embedding、注意力和 MLM 头。
    if epoch == 0:  # 首轮记录实际 token embedding 梯度规模。
        first_gradient_norm = bert.token_embedding.weight.grad.norm().item()  # 读取 embedding 表的首轮梯度范数。
    optimizer.step()  # 根据当前梯度更新全部 BERT 参数。
    loss_trace.append(loss.item())  # 保存当前 MLM 损失。
    if epoch in [0, 50, 150, 400]:  # 选择关键轮次展示训练轨迹。
        current_accuracy = (masked_logits.argmax(dim=1) == mlm_targets).float().mean().item()  # 计算当前 MASK 位置准确率。
        print(f"epoch={epoch:03d} loss={loss.item():.4f} MLM_accuracy={current_accuracy:.1%}")  # 输出真实损失和 MLM 准确率。
bert.eval()  # 切换到评估模式获取稳定结果。
with torch.no_grad():  # 关闭评估阶段的梯度记录。
    final_logits, final_attention, final_encoded = bert(masked_ids)  # 重新计算最终词表 logits 与双向注意力。
final_masked_logits = final_logits[:, mask_position, :]  # 提取八个 MASK 位置的最终 logits。
final_predictions = final_masked_logits.argmax(dim=1)  # 选择每条工单概率最大的预测 token。
mlm_accuracy = (final_predictions == mlm_targets).float().mean().item()  # 计算手写 BERT 的 MLM 准确率。
print(f"首轮 token embedding 梯度范数={first_gradient_norm:.6f}")  # 输出非零梯度证明模型被真实训练。
first_mask_attention = final_attention[0, :, mask_position, :].mean(dim=0)  # 对第一条工单各头的 MASK 注意力取平均。
print("第一条工单 MASK 对各位置的平均注意力：")  # 标记即将展示的双向注意力分布。
for position, token in enumerate(["工单", "用户", "无法", "[MASK]", "请", "重置", "密码"]):  # 逐位置展示左右上下文权重。
    print(f"  位置{position} {token:<6} 权重={first_mask_attention[position].item():.4f}")  # 输出当前上下文 token 的平均注意力。

epoch=000 loss=3.8516 MLM_accuracy=12.5%


epoch=050 loss=0.0002 MLM_accuracy=100.0%


epoch=150 loss=0.0001 MLM_accuracy=100.0%


epoch=400 loss=0.0000 MLM_accuracy=100.0%
首轮 token embedding 梯度范数=0.079752
第一条工单 MASK 对各位置的平均注意力：
  位置0 工单     权重=0.0000
  位置1 用户     权重=0.0000
  位置2 无法     权重=0.5000
  位置3 [MASK] 权重=0.2501
  位置4 请      权重=0.0000
  位置5 重置     权重=0.2499
  位置6 密码     权重=0.0000


## 结果解读：每个 MASK 的候选 logits

输出前三候选可以检查模型究竟是在恢复故障词，还是只给所有样本同一个高频答案。第一条注意力同时覆盖 MASK 左右两侧，验证这里没有错误地套用 GPT 因果 mask。

In [5]:
print("工单上下文                         目标   预测   top3(token, logit)")  # 打印逐工单 MLM 结果表头。
for index, sequence in enumerate(ticket_sequences):  # 遍历八条工单展示真实候选排名。
    top_values, top_indices = torch.topk(final_masked_logits[index], k=3)  # 取得当前 MASK 位置的前三词表 logits。
    top_candidates = [(id_to_token[token_id.item()], round(value.item(), 3)) for value, token_id in zip(top_values, top_indices)]  # 把候选编号转换成可读 token。
    context = "/".join(sequence[:mask_position] + ["[MASK]"] + sequence[mask_position + 1:])  # 构造当前遮盖工单的可读上下文。
    print(f"{context:<34}  {id_to_token[mlm_targets[index].item()]:<4}  {id_to_token[final_predictions[index].item()]:<4}  {top_candidates}")  # 输出目标、预测和前三候选。
print(f"同 MASK 位置准确率：最高频基线={baseline_accuracy:.1%}，手写 BERT MLM={mlm_accuracy:.1%}")  # 汇总同数据和指标下的方案对比。

工单上下文                         目标   预测   top3(token, logit)
工单/用户/无法/[MASK]/请/重置/密码             登录    登录    [('登录', 17.577), ('待发', 5.854), ('扣款', 5.77)]
工单/账号/突然/[MASK]/请/核验/身份             锁定    锁定    [('锁定', 17.443), ('扣款', 6.311), ('错误', 5.713)]
工单/订单/一直/[MASK]/请/查询/物流             待发    待发    [('待发', 19.152), ('缺货', 7.248), ('错误', 7.025)]
工单/包裹/到货/[MASK]/请/查询/物流             延迟    延迟    [('延迟', 15.265), ('登录', 3.651), ('使用', 2.927)]
工单/电子/发票/[MASK]/请/修改/抬头             错误    错误    [('错误', 18.701), ('锁定', 6.754), ('待发', 6.669)]
工单/商品/显示/[MASK]/请/检查/库存             缺货    缺货    [('缺货', 15.412), ('错误', 4.014), ('待发', 3.879)]
工单/支付/重复/[MASK]/请/发起/退款             扣款    扣款    [('扣款', 16.846), ('登录', 5.308), ('锁定', 5.097)]
工单/优惠/无法/[MASK]/请/核对/活动             使用    使用    [('使用', 13.991), ('缺货', 2.345), ('延迟', 1.989)]
同 MASK 位置准确率：最高频基线=12.5%，手写 BERT MLM=100.0%


## 失败案例：未真正替换原词导致标签泄漏

如果数据管道把标签位置原样留在输入，最简单的“复制当前位置 token”就能得到 100%，但部署时输入只有 `[MASK]` 会立刻失效。下面不用训练模型也能精确复现这个虚假指标。

In [6]:
leaky_copy_predictions = original_ids[:, mask_position]  # 模拟从未遮盖输入直接复制目标位置的错误捷径。
proper_copy_predictions = masked_ids[:, mask_position]  # 在真正遮盖输入上执行同一个复制策略。
leaky_copy_accuracy = (leaky_copy_predictions == mlm_targets).float().mean().item()  # 计算泄漏数据上的虚假准确率。
proper_copy_accuracy = (proper_copy_predictions == mlm_targets).float().mean().item()  # 计算正确遮盖数据上的复制策略准确率。
print(f"错误数据管道：当前位置仍是原词，复制准确率={leaky_copy_accuracy:.1%}")  # 展示标签泄漏制造的完美假象。
print(f"修复数据管道：当前位置替换为 [MASK]，复制准确率={proper_copy_accuracy:.1%}")  # 展示正确输入消除了复制捷径。
print(f"真正 MLM 模型在遮盖输入上的准确率={mlm_accuracy:.1%}")  # 对照模型确实利用上下文恢复目标。
print("修复结论：在进入模型前抽查 input_ids 与 labels，绝不能让目标原词残留。")  # 总结 MLM 数据管道的必要门禁。

错误数据管道：当前位置仍是原词，复制准确率=100.0%
修复数据管道：当前位置替换为 [MASK]，复制准确率=0.0%
真正 MLM 模型在遮盖输入上的准确率=100.0%
修复结论：在进入模型前抽查 input_ids 与 labels，绝不能让目标原词残留。


## 生产差距

真实 BERT 预训练会动态抽样多个 MASK，采用 80/10/10 替换策略、海量语料、多层 Encoder、分布式优化和独立验证集。本例只实现 MLM，不实现 NSP；若业务需要句间关系，应明确增加 NSP、SOP 或对比目标，而不能把单句 MLM 指标冒充句间能力。还要监控数据去重、敏感信息、词表覆盖和下游迁移。

## 最小回归测试

In [7]:
assert len(ticket_sequences) >= 6  # 保证案例覆盖足够多的真实客服工单语义。
assert loss_trace[-1] < loss_trace[0]  # 保证真实反向传播使 MLM 损失下降。
assert first_gradient_norm > 0.0  # 保证 token embedding 获得了非零梯度。
assert mlm_accuracy > baseline_accuracy  # 保证手写 Encoder 在同一 MASK 指标上超过高频基线。
assert mlm_accuracy == 1.0  # 保证教学模型已恢复全部八个遮盖 token。
assert torch.isclose(first_mask_attention.sum(), torch.tensor(1.0), atol=1e-5)  # 保证 MASK 注意力在全序列位置上归一化。
assert leaky_copy_accuracy == 1.0 and proper_copy_accuracy == 0.0  # 保证标签泄漏失败与遮盖修复均可复现。
print("回归测试通过：双向注意力、MLM 训练和泄漏门禁均符合预期。")  # 输出集中断言的最终验收结果。

回归测试通过：双向注意力、MLM 训练和泄漏门禁均符合预期。
